# TPUForge Multi-Chip Kaggle TPU v5e-8 & Cloud TPU v6e/v7 Benchmark

This notebook builds the native C++20 `tpuforge` runtime (`_tpuforge_ext` + `tpuforge_bench`) and benchmarks **Vanilla JAX**, **StaticArena (`M-002`)**, and **TPUForge Tree Speculation (`M-002 + M-003`)** on real **Qwen2.5-Coder** weights.

### Kaggle Free TPU `v5e-8` & Cloud TPU Support
- **8x TPU v5e chips (`128 GB` pooled HBM):** Runs `Qwen/Qwen2.5-Coder-7B-Instruct` and fits `Qwen/Qwen2.5-Coder-32B-Instruct` via **OpenXLA GSPMD Megatron Tensor Parallelism (`TP=8`)** + streaming shard-by-shard `.safetensors` loading (`~2.3 GB` peak host RAM per weight stack).
- **Dual Systolic Tile Widths (`128` vs `256`):** Auto-infers `128x128` on TPU v2-v5e and `256x256` on TPU v6e Trillium / v7 Ironwood / TPU 8x via `tpu.infer_mxu_dim(device_kind)`.
- **Kaggle Cache Rule:** Large Hugging Face weights are cached in `HF_HOME=/tmp/hf_cache` (tmpfs/scratch) so `/kaggle/working` stays free for JSON benchmark telemetry.
- **Preinstalled JAX/libtpu Preserved:** Only `cmake`, `ninja`, `nanobind`, `huggingface_hub`, `tokenizers`, and `ml_dtypes` are checked/installed.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ.setdefault("HF_HOME", "/tmp/hf_cache")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.makedirs("/tmp/hf_cache", exist_ok=True)

RESULTS_DIR = Path("/kaggle/working/tpuforge_results") if Path("/kaggle/working").exists() else Path("./tpuforge_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Locate uploaded bundle zip or clone repo
REPO_DIR = Path("/tmp/TPUForge")
bundle_candidates = [
    Path("/tmp/tpuforge_kaggle_bundle.zip"),
    Path("/kaggle/input/tpuforge-src/tpuforge_kaggle_bundle.zip"),
    Path("/kaggle/working/tpuforge_kaggle_bundle.zip"),
    Path("./tpuforge_kaggle_bundle.zip"),
]
bundle_zip = next((p for p in bundle_candidates if p.is_file()), None)

if bundle_zip is not None:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_zip, "r") as zf:
        zf.extractall(REPO_DIR)
    print(f"Extracted TPUForge bundle from {bundle_zip} -> {REPO_DIR}")
elif Path("CMakeLists.txt").is_file() and Path("src/speculative/py_bindings.cpp").is_file():
    REPO_DIR = Path(".").resolve()
    print(f"Using current working checkout: {REPO_DIR}")
else:
    repo_url = os.environ.get("TPUFORGE_REPO_URL", "")
    if not repo_url:
        raise FileNotFoundError(
            "Upload tpuforge_kaggle_bundle.zip as a Kaggle Dataset or set TPUFORGE_REPO_URL."
        )
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", repo_url, str(REPO_DIR)], check=True)

# Install build dependencies without touching Kaggle's preinstalled jax / libtpu
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "cmake", "ninja", "nanobind", "huggingface_hub", "tokenizers", "ml_dtypes"
], check=True)

build_dir = REPO_DIR / "build"
subprocess.run([
    "cmake", "-S", str(REPO_DIR), "-B", str(build_dir),
    "-G", "Ninja", "-DCMAKE_BUILD_TYPE=Release",
    f"-DPython_EXECUTABLE={sys.executable}",
], check=True)
subprocess.run(["cmake", "--build", str(build_dir)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(build_dir) not in sys.path:
    sys.path.insert(0, str(build_dir))

import jax
import _tpuforge_ext as tpu
devs = jax.devices()
dev_kind = str(getattr(devs[0], "device_kind", devs[0]))
print(f"JAX backend={jax.default_backend()} | devices={len(devs)}x {dev_kind} | auto MXU={tpu.infer_mxu_dim(dev_kind)}")

In [ ]:
# Benchmark 1: Qwen2.5-Coder-7B-Instruct (auto TP=4 on v5e-8, auto tile 128 on v5e or 256 on v6e/v7)
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "benchmarks.run_qwen_bench",
    "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
    "--max-seq-len", "2048",
    "--max-new-tokens", "128",
    "--tile-size", "auto",
    "--tp-size", "auto",
    "--output-json", str(RESULTS_DIR / "qwen_7b_bench.json"),
], check=True)

In [ ]:
# Benchmark 2: Qwen2.5-Coder-32B-Instruct via Megatron TP=8 across Kaggle v5e-8 (128 GB HBM pool)
import jax
import subprocess
import sys

if len(jax.devices()) >= 8:
    subprocess.run([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", "Qwen/Qwen2.5-Coder-32B-Instruct",
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", "auto",
        "--tp-size", "8",
        "--output-json", str(RESULTS_DIR / "qwen_32b_tp8_bench.json"),
    ], check=True)
else:
    print(f"Skipping 32B TP=8 benchmark: only {len(jax.devices())} device(s) visible (needs >= 8).")

In [ ]:
# Benchmark 3: Systolic Tile Width Comparison (128x128 vs 256x256)
import json
import subprocess
import sys

COMPARE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
reports = {}
for tile in ("128", "256"):
    out_file = RESULTS_DIR / f"qwen_tile_{tile}.json"
    subprocess.run([
        sys.executable, "-m", "benchmarks.run_qwen_bench",
        "--model", COMPARE_MODEL,
        "--max-seq-len", "2048",
        "--max-new-tokens", "128",
        "--tile-size", tile,
        "--tp-size", "auto",
        "--output-json", str(out_file),
    ], check=True)
    reports[tile] = json.loads(out_file.read_text(encoding="utf-8"))

print("\n=== Systolic Tile Comparison (Workload 2: Code Refactor) ===")
print(f"{'Tile':<6} | {'TP':<3} | {'Vanilla tok/s':<13} | {'TPUForge tok/s':<14} | {'Tok/Pass':<8} | {'Vanilla Compiles':<16}")
for tile, rep in reports.items():
    w2 = rep["workload_2_code_refactor"]
    w1 = rep["workload_1_prompt_sweep"]
    print(
        f"{rep['tile_size']:<6} | {rep['tp_size']:<3} | "
        f"{w2['vanilla_jax']['decode_tok_per_s']:<13.2f} | "
        f"{w2['tpuforge_tree']['decode_tok_per_s']:<14.2f} | "
        f"{w2['tpuforge_tree']['mean_tokens_per_decode_pass']:<8.2f} | "
        f"{w1['vanilla_compiles']:<16}"
    )

In [ ]:
# Benchmark 4: Milestone M-005 Optical Torus Ring-Attention over ICI (prefill + stationary-KV systolic tree)
import jax
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "benchmarks.ring_attention_jax",
    "--max-seq-len", "32768",
    "--valid-seq-len", "28672",
    "--ring-size", "auto",
    "--tp-size", "1",
    "--tile-size", "auto",
    "--output-json", str(RESULTS_DIR / "m005_ring_attention.json"),
], check=True)

if len(jax.devices()) >= 8:
    subprocess.run([
        sys.executable, "-m", "benchmarks.ring_attention_jax",
        "--max-seq-len", "131072",
        "--valid-seq-len", "114688",
        "--ring-size", "8",
        "--tp-size", "1",
        "--tile-size", "auto",
        "--no-check-parity",
        "--output-json", str(RESULTS_DIR / "m005_ring_128k.json"),
    ], check=True)
else:
    print(f"Skipping 128k 8-chip ring run: only {len(jax.devices())} device(s) visible (needs >= 8).")